In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams["figure.autolayout"] = True # to for tight_layout()
import pymc as pm
import arviz as az
import scipy as sp
from scipy.special import expit as logistic

# Announcements

## Create/Join group for the exam by April 28

* We facilitate group formation in LearnIT: [Preliminary Exam groups](https://learnit.itu.dk/mod/groupselect/view.php?id=246486)

<br>

* **Create/join must be formed by April 28.** Contact Oleg (`olejar@itu.dk`) if you need help finding a group

<br>

* Group formation is also enabled in WISEflow – ITU's new exam system
    * **You must replicate the LearnIT group in WISEflow to submit your exam**

<br>

* Groups must be of size 2 or 3 members
    * All members must belong to the same study programme


## Exam publication next week

* The second half of the lecture will be about presenting the exam and you will have an opportunity to ask any questions
    * Please review the instructions for submitting the report in previous year exams before the lecture next week
    * The instructions for this year are very similar, so you can bring up any questions in the next lecture

<br>

* After the lecture we will not answer any questions about this year's exam, as the examination is officially ongoing at that point

<br>

* During the exercise session you can only ask about previous exercises
    * Oleg will discuss the solution of an exam from previous years

<br>

## Please fill in the course evaluation

<br><br><br><br><br><br>

# Multilevel Models

## Agenda

* Our first multilevel model
* {Under/Over}fitting in multilevel models
* More than one cluster
* Divergences and non-centered models
* Multilevel posterior predictive checks

<br><br><br><br><br><br><br>

## Intro

* The models we have considered so far treat each group/cluster of data independently
    * What the model learns for one group is not taken into account for others

<br>

* Although it is correct to analyze data from different groups with different parameters, it might also be useful to learn parameters _across groups_ (for the whole population)


<br>



* Some benefits of multilevel models:
    1. Improved estimates for repeat sampling
        * Prevents the over/underfitting of single-level models
    2. Improved estimates for imbalance in sampling
        * Cope with different uncertainty across groups
    3. Estimates of variation
        * Model variation across groups explicitly
    4. Avoid averaging, retain variation
        * Data does not need to be averaged, variation among groups can captured by the model


<br>


* Some disadvantages of multilevel models:
    * Sampling is harder
    * Interpretation of certain parameters can be cumbersome
    * Standard comparison metrics such as WAIC become more subtle to understand
    * Posterior predictive checks are more complicated


<br>
    
    
* Often multilevel models are known as _hierarchical models_


<br><br><br>

## Reed Frog survival

<img src="figs/tadpole.jpg" style="margin: 5px" width=300px align=right>

* We will estimate the rate of survival of Reed-frogs tadpoles (*Hyperolius spinigularis*)

<br>

* The tadpoles are placed in different "tanks" (containers) with different number of tadpoles
    * As we will observe, there is a lot of dispersion in the data (survival rate differs from tank to tank)

<br>

* The dataset is loaded below, we focus on the following variables
    * `density` - number of frogs in the container
    * `surv` - number of frogs that survived
    
<br>

* Each row in the dataset records the results of a specific tank

In [ ]:
d = pd.read_csv("Data/reedfrogs.csv", sep=";")
print(d.shape)
d.head()

* Our goal is to estimate the rate of survival in each container


* First, we use a single-level model

\begin{align*}
S_i &\sim \mathrm{Binomial}(N_i,p_i) \\
\mathrm{logit}(p_i) &= \alpha_{\mathrm{TANK}[i]} \\
\alpha_j &\sim \mathrm{Normal}(0,1.5)
\end{align*}


In [ ]:
with pm.Model() as m13_1:
    α   = pm.Normal('α',mu=0,sigma=1.5,shape=d.shape[0])
    p   = pm.Deterministic('p',pm.math.invlogit(α))
    obs = pm.Binomial('obs',n=d.density,p=p,observed=d.surv)

In [ ]:
pm.model_to_graphviz(m13_1)

In [ ]:
trace_13_1 = pm.sample(model=m13_1,return_inferencedata=True,idata_kwargs=dict(log_likelihood=True))

In [ ]:
pm.summary(trace_13_1)

In [ ]:
pm.plot_forest(trace_13_1, var_names=['α'], transform=logistic, combined=True, hdi_prob=.95);

### First multilevel model

* Now extend the model above with two population-level parameters
    * $\bar{\alpha}$ - survival rate across tanks (for the complete population of frogs)
    * $\sigma$ - dispersion across tanks (for the complete population of frogs)
    
<br>

* The parameters are introduced to the model as follows


\begin{align*}
S_i &\sim \mathrm{Binomial}(N_i,p_i) \\
\mathrm{logit}(p_i) &= \alpha_{\mathrm{TANK}[i]} \\
\alpha_j &\sim \mathrm{Normal}(\bar{\alpha},\sigma) ~~~~ \text{ note that } \bar{\alpha},\sigma \text{ are shared across tanks}\\
\bar{\alpha} &\sim \mathrm{Normal}(0,1.5) \\
\sigma &\sim \mathrm{Exponential}(1)
\end{align*}

<br>

* This type of multilevel model is called *varying intercepts*

<br>

* It is important to note that now information learned for tank also informs the others (through the shared parameters $\bar{\alpha}$ and $\sigma$)

<br>

* The high-level parameters ($\bar{\alpha}, \sigma$) are known as _hyper-parameters_ and their priors _hyper-priors_

<br>

* The posterior for this model includes all levels at the same time

<br>

In [ ]:
tank = np.arange(d.shape[0])
with pm.Model() as m13_2:
    σ     = pm.Exponential('σ',lam=1.0)
    α_bar = pm.Normal('α_bar',mu=0.0,sigma=1.5)
    α     = pm.Normal('α',mu=α_bar,sigma=σ,shape=d.shape[0])
    p     = pm.Deterministic('p',pm.math.invlogit(α))
    obs   = pm.Binomial('obs',n=d.density,p=p,observed=d.surv)    

In [ ]:
pm.model_to_graphviz(m13_2)

In [ ]:
trace_13_2 = pm.sample(model=m13_2,return_inferencedata=True,idata_kwargs=dict(log_likelihood=True))

In [ ]:
pm.summary(trace_13_2,var_names=['α_bar','σ','α'],round_to=2)

* The posterior mean of $\sigma$ is about 1.6 (slightly strong prior)
    * This is an instance of a regularizing prior, although it has been learned from the data

<br>

* Now we compare the information criteria of the single-level and multilevel models

In [ ]:
pm.compare({'m13_1': trace_13_1, 'm13_2': trace_13_2}, ic='waic', scale='deviance')

* Note the reduced value of `p_waic`, 21 effective parameters out of 50 that the model features
    * This is due to the regularizing effect of $\bar{\alpha}$ and $\sigma$ on $\alpha_j$

### Shrinkage (pooling)

* Shrinkage refers to the effect of population level parameters in lower level parameters
    * It is a consequence of the multilevel model
    * Samples of low level parameters depend on the values of high-level parameters
    
<br>


* Shrinkage is useful when some of the groups (tanks) are underrepresented in the dataset

<br>

* In the plot below illustrates the effect of shrinkage on the samples
    * Blue dots are the empirical estimates of survival rate
    * White dots are the inferred estimates in the multilevel model
    * The dashed line is the mean of the population level survival probability
    * Points are ordered from tanks with low density of frogs to higher density

<br>

* We observe stronger shrinkage for small tanks, as the likelihood makes them contribute less to the posterior 
    * White points for small tanks are more "pulled" towards the dashed line (average estimate)
    * White points for small tanks are further away from blue points (empirical estimate)
<br>

In [ ]:
est = trace_13_2.posterior.p.mean(['chain','draw'])
x = np.arange(1,49)
plt.subplots(figsize=(9,4))
plt.hlines(xmin=0,xmax=48,y=logistic(trace_13_2.posterior.α_bar.mean()),linestyles='--',color='k')
plt.vlines(ymin=0,ymax=1,x=16.5,linestyles='-',color='k',lw=.5)
plt.vlines(ymin=0,ymax=1,x=32.5,linestyles='-',color='k',lw=.5)
plt.scatter(x,est,facecolor='w',edgecolors='k')
plt.scatter(x, d.propsurv)
plt.text(8, 0, "small tanks", horizontalalignment="center")
plt.text(16 + 8, 0, "medium tanks", horizontalalignment="center")
plt.text(32 + 8, 0, "large tanks", horizontalalignment="center");

* We may use the posterior samples to visualize the distribution of population level log-odds ($\bar{\alpha}$)
    * Below we plot the log-odds distribution from $100$ posterior samples
    * This plot gives visual information on the variance and shape of the population level log-odds


<br>


* Additionally, we simulate survival probabilities (for new hypothetical tanks)
    * Below we plot the survival probability for 4000 new tanks

In [ ]:
_, ax = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)

trace_13_2_α_bar = trace_13_2.posterior.α_bar.values.reshape(4000)
trace_13_2_σ     = trace_13_2.posterior.σ.values.reshape(4000)

# show first 100 populations in the posterior
num_samples = 100
xrange = np.linspace(-3, 4, 200)
postcurve = [
    sp.stats.norm.pdf(xrange, loc=trace_13_2_α_bar[i], scale=trace_13_2_σ[i])
    for i in range(num_samples)
]
ax[0].plot(xrange, np.asarray(postcurve).T, alpha=0.1, color="k")
ax[0].set_xlabel("log-odds survive")
ax[0].set_ylabel("Density")


# sample 4000 imaginary tanks from the posterior distribution
sim_tanks = np.random.normal(loc=trace_13_2_α_bar, scale=trace_13_2_σ)

# transform to probability and visualize
pm.plot_kde(logistic(sim_tanks), ax=ax[1], plot_kwargs={"color": "k"})
ax[1].set_xlabel("probability survive")
ax[1].set_ylabel("Density");

### Multilevel models are mixtures

* The left-plot above is similar than the plot of posterior beta distributions in the $\textrm{Beta-Binomial}$ mixture model in the previous chapter

* Since multilevel models are mixtures, they are appropriate to handle over-dispersion

## Under/Overfitting in multilevel models


* Pooling (shrinkage) is directly related to under/overfitting

<br>

* We can look at this effect from three perspectives (related to the frogs example):

    1. Complete pooling
        * We assume that the survival probability is the same for all tanks, so we use a single parameter for all data
        * This approach tends to _underfits_ the data
    2. No pooling
        * We assume that the survival probability is different for each tank, so we use a different parameter for each tank
        * This approach tends to _overfits_ the data
    3. Partial pooling
        * We assume that the survival is different for each tank (due to unmeasured conditions in the tank), but there is a global survival probability for frogs (which we model using high level parameters)
        * Provides a better trade-off between under/overfitting

<br>

<div style="width:400px;  height: 50px; display:flex; align-items:center; justify-content:center; background:blue; text-align:center; box-sizing:border-box; margin:8px auto; color: white">
    <strong style="font-size: 16px">
        Why does "complete pooling" underfit data?<br>
        Why does "no pooling" overfit data?
    </strong>
</div>

<br>

* We illustrate the points above with an example on simulated data

### Simulated frogs data in ponds

* The code below creates a dataset similar to `reedfrogs.csv` but with known $\bar{\alpha}$ and $\alpha$

<br>

* Then we apply the multilevel model below to estimate the parameters, and check how far they are from the values used to generate the data

<br>

* We compare the results with the no-pooling approach
    * To this end, we compare the results of the models with the empirical estimates from the generated dataset

In [ ]:
# parameters to recover
α_bar  = 1.5
σ      = 1.5

# number of ponds
nponds = 60

# sample densities per pond
Ni     = np.repeat([5,10,25,35],15)

# sample log-odds per ponds
α_pond = np.random.normal(loc=α_bar,scale=σ,size=nponds)

# creating dataset
dsim = pd.DataFrame(dict(pond=np.arange(nponds), Ni=Ni, true_α=α_pond))

# simulate number of frogs that survived
dsim['Si']       = sp.stats.binom(n=dsim.Ni,p=logistic(dsim.true_α)).rvs()
# no-pooling estimates (for comparison)
dsim['p_nopool'] = dsim.Si/dsim.Ni

dsim.head()

* Below is the previous multilevel model applied to the new dataset

In [ ]:
with pm.Model() as m13_3:
    σ      = pm.Exponential('σ',lam=1.0)
    α_bar  = pm.Normal('α_bar',mu=0.0,sigma=1.5)
    α_pond = pm.Normal('α_pond',mu=α_bar,sigma=σ,shape=dsim.shape[0])
    p_pond = pm.Deterministic('p_pond',pm.math.invlogit(α_pond))
    obs    = pm.Binomial('obs',n=dsim.Ni,p=p_pond,observed=dsim.Si)

In [ ]:
trace_13_3 = pm.sample(model=m13_3,return_inferencedata=True)

* We observe that the estimated values for $\bar{\alpha}$ and $\sigma$ are very close to the values we use to generate the data

* This indicates that the model correctly estimates the values for these parameters in the model

In [ ]:
pm.summary(trace_13_3,var_names=['α_bar','σ'],round_to=2)

* We compute the distribution of absolute error using the posterior samples
    * Concretely, we compute the average survival proportion for each pond using posterior samples

<br>


* This will help to visualize the error for different ponds sizes
    * Actually we observe the absolute error for each data point
    
    


In [ ]:
dsim['p_partpool'] = trace_13_3.posterior.p_pond.mean(['chain','draw'])
dsim['p_true'] = logistic(dsim['true_α'].values)

dsim['nopool_error'] = np.abs(dsim.p_nopool - dsim.p_true)
dsim['partpool_error'] = np.abs(dsim.p_partpool - dsim.p_true)

dsim.head()

In [ ]:
_, ax = plt.subplots(1, 1, figsize=(12, 5))
xrange = np.arange(60)
xrange_ = xrange.reshape((4, 15))

ax.scatter(xrange + 1, dsim.nopool_error, alpha=0.6)
ax.scatter(xrange + 1, dsim.partpool_error, facecolors="none", edgecolors="k", lw=1.2)
ax.vlines(xrange_[1:, 0] + 0.5, -0.025, max(dsim.nopool_error) + 0.05, lw=0.5)

textall = [
    "tiny ponds (5)",
    "small ponds (10)",
    "medium ponds (25)",
    "large ponds (35)",
]
for isem in range(4):
    ax.hlines(
        dsim.nopool_error[xrange_[isem, :]].mean(),
        xrange_[isem, 0] + 1,
        xrange_[isem, -1] + 1,
        color="C0",
        alpha=0.6,
    )
    ax.hlines(
        dsim.partpool_error[xrange_[isem, :]].mean(),
        xrange_[isem, 0] + 1,
        xrange_[isem, -1] + 1,
        color="k",
        linestyles="--",
    )
    ax.text(
        xrange_[isem, 7] + 0.5,
        max(dsim.nopool_error) + 0.05,
        textall[isem],
        horizontalalignment="center",
    )

ax.set_xlabel("pond")
ax.set_ylabel("absolute error")
ax.set_xlim(-1, 62);

* The plot above (should) show the dashed line below the blue line for most cases
    * That is, on average, the partial pooling model has lower absolute error

<br>

* On average, lower size ponds have higher errors

<br>

* On average, posterior estimates (white points) have absolute lower error

<br>

* In general, partial pooling helps in the presence of low data clusters; their estimates get information from the population level parameters.

<br>

* For large clusters, partial pooling shows a lower effect,
    * In general, it does not significantly degrade the accuracy of the estimates

<br>

* In the presence of (real) outliers, partial pooling might decrease accuracy


## More than one type of group/cluster


* Multilevel models do necessary need to apply to single parameter

<br>

* Now we apply a multilevel model with _two_ high-level parameters
    * One for the actor (the chimpanzee involved in the experiment)
    * Another one for the block the chimpanzee belongs

In [ ]:
d_ch = pd.read_csv("Data/chimpanzees.csv", sep=";")

treatment = (d_ch.prosoc_left + 2 * d_ch.condition).values
Ntreatments = len(np.unique(treatment))

actor = (d_ch.actor - 1).astype(int).values
Nactor = len(np.unique(actor))

block = (d_ch.block - 1).astype(int).values
Nblock = len(np.unique(block))

* The model only requires the addition of three high level parameters

\begin{align*}
L_i &\sim \mathrm{Binomial}(1,p_i) \\
\mathrm{logit}(p_i) &= \alpha_{\mathrm{ACTOR}[i]} + \gamma_{\mathrm{BLOCK}[i]} + \beta_{\mathrm{TREATMENT}[i]}\\
\beta_j &\sim \mathrm{Normal}(0,0.5) ~~ \text{ for }j \in \{1,\ldots,4\} \\
\alpha_k &\sim \mathrm{Normal}(\bar{\alpha},\sigma_\alpha) ~~~ \text{ for }k \in \{1,\ldots,7\} \\
\gamma_l &\sim \mathrm{Normal}(0,\sigma_\gamma) ~~ \,\, \text{ for }l \in \{1,\ldots,6\} \\
\bar{\alpha} &\sim \mathrm{Normal}(0,1.5) \\
\sigma_\alpha &\sim \mathrm{Exponential}(1) \\
\sigma_\gamma &\sim \mathrm{Exponential}(1) 
\end{align*}

* The $\sigma_\alpha$ and $\sigma_\gamma$ control/determine the amount of pooling between chimpanzees and blocks

* Note that there is no $\bar{\gamma}$, this is to avoid high correlation between $\alpha_k$ and $\gamma_l$ parameters (recall the right/left leg model)

In [ ]:
with pm.Model() as m13_4:
    σ_α = pm.Exponential('σ_α',lam=1.0)
    σ_γ = pm.Exponential('σ_γ',lam=1.0)
    α_bar = pm.Normal('α_bar',mu=0,sigma=1.5)
    
    γ = pm.Normal('γ',mu=0,sigma=σ_γ,shape=Nblock)
    α = pm.Normal('α',mu=α_bar,sigma=σ_α,shape=Nactor)
    β = pm.Normal('β',mu=0,sigma=0.5,shape=Ntreatments)
    
    p = pm.math.invlogit(α[actor] + γ[block] + β[treatment])
    
    L = pm.Binomial('L',n=1,p=p,observed=d_ch.pulled_left)

In [ ]:
pm.model_to_graphviz(m13_4)

#### Divergences in the chimpanzees model

In [ ]:
trace_13_4 = pm.sample(model=m13_4,idata_kwargs=dict(log_likelihood=True))

* Note the warning about divergent transitions
    * We will discuss this issue in more detail in the next section

In [ ]:
pm.summary(trace_13_4, var_names=['γ', 'α', 'β', 'σ_α', 'σ_γ'], round_to=2)

* The posterior shows more variation on the actor parameter ($\alpha$) than the block parameter ($\gamma$)


<br>


* This can be nicely visualized in a forest plot
    * Note that the width of the HDI for $\alpha_k$ is larger on average than for $\gamma_l$
    * The high level parameters $\sigma_\alpha$ and $\sigma_\gamma$ also show this difference
    

<br>


* These results indicate that adding block as a predictor does not introduce overfitting

In [ ]:
fig, (ax1, ax2) = plt.subplots(1,2, figsize=(12,5))

pm.plot_forest(trace_13_4,combined=True,var_names=['β','α','γ','α_bar','σ_α','σ_γ'],hdi_prob=.89, ax=ax1);

pm.plot_dist(trace_13_4.posterior['σ_α'],color='k',label='Actor', ax=ax2);
pm.plot_dist(trace_13_4.posterior['σ_γ'],color='b',label='Block', ax=ax2);

In [ ]:
with pm.Model() as m13_5:
    σ_α = pm.Exponential('σ_α',lam=1.0)
    α_bar = pm.Normal('α_bar',mu=0,sigma=1.5)
    
    α = pm.Normal('α',mu=α_bar,sigma=σ_α,shape=Nactor)
    β = pm.Normal('β',mu=0,sigma=0.5,shape=Ntreatments)
    
    p = pm.math.invlogit(α[actor] + β[treatment])
    
    L = pm.Binomial('L',n=1,p=p,observed=d_ch.pulled_left)

In [ ]:
trace_13_5 = pm.sample(model=m13_5,return_inferencedata=True,idata_kwargs=dict(log_likelihood=True))

* Now we compare the multilevel models with and without the block predictor


* Although block introduces 7 more parameters ($\gamma_l$), the effective number of parameters is 2 less than for the model without block
    * This is because $\gamma_l$ parameters are very close to zero a do not have a strong impact on the output

In [ ]:
pm.compare({'m13.4': trace_13_4, 'm13.5': trace_13_5},ic='waic',scale='deviance')

## Divergent transitions (and how to fix them)

* Divergent transitions is a problem inherent to Hamiltonian Monte Carlo (HMC)
    * But it is an advantage of HMC compared to other MCMC methods; as it has built-in sampling diagnostic embedded

<br>


* Recall that HMC runs a simulation of Hamiltonian physics


<br>


* The simulation must be _discretized_ in a series of leapfrog steps
    * As a consequence, we are introducing a small error at each step
    * Typically, this error is small

<br>


* HMC can detect whether the error in the simulation is too large
    * Recall in the simulation the sum of potential and total energy must the same at the beginning and the end of the simulation
    $$
    H(q, p) = U(q) + K(p)
    $$
    * When the errors are introduced in the simulation the equation above doesn't hold at in the final state


<br>

    
* HMC algorithms discard divergent transitions, and, consequently, reduces the sampling performance


<br>


* Divergent transitions appear in the high-curvature areas of the posterior
    * These areas are difficult to discretize 
    * They result in final states far from actual posterior

<br>
    
* There are two common solutions to reduce the amount of divergent transitions
    * Tuning the simulation, e.g., by increasing the target_accept during warm up 
    * Reparameterize the model. We will see what this means soon
 
<br>

### Funnel example

* We will illustrate divergent transitions with a characteristic posterior with high curvature
    
<br>


* The model, which is very simple, is defined as follows

\begin{align*}
v &\sim \mathrm{Normal}(0,3) \\
x &\sim \mathrm{Normal}(0,\exp(v)) 
\end{align*}

In [ ]:
with pm.Model() as m13_7:
    v = pm.Normal('v',mu=0,sigma=3)
    x = pm.Normal('x',mu=0,sigma=pm.math.exp(v))

In [ ]:
trace_13_7 = pm.sample(model=m13_7,return_inferencedata=True)

* Note the high number of divergent transitions reported by PyMC


<br>


* Below we show the diagnosis 
    * The ESS is very low for all parameters
    * The black lines in the trace plots mark the divergent transitions


<br>

In [ ]:
pm.summary(trace_13_7,round_to=2)

In [ ]:
pm.plot_trace(trace_13_7);

* The plot below show the shape of the posterior and marks all divergent transitions


* We can observe how an area with very high density (a deep valley in the negative log posterior) with many divergent transitions

In [ ]:
pm.plot_pair(trace_13_7,var_names=['x','v'],divergences=True);

### Non-centered models

* The model above is an instance of a _centered_ model    
    * For each value of $v$ the distribution of $x$ changes drastically, making hard for the sampler
    
<br>


* The divergences warning messages in PyMC recommends to reparameterize the model


<br>


* The message above hints at using a _non-centered_ model
    * We would like to avoid the drastic changes in distribution
    

<br>


* An example of this type of reparametization is combining the parameters by means of an arithmetic operation (instead having one embedded in the other)


<br>


* The funnel model above can be rewritten as follows

\begin{align*}
v &\sim \mathrm{Normal}(0,3) \\
z_x &\sim \mathrm{Normal}(0,1) \\
x &= z_x\exp(v)
\end{align*}


<br>


* The model above simply uses a fact about standardized data
    * Recall that when data is standardized it has mean 0 and standard deviation 1
    * We use the variable $z_x$ above as the standardized version of $x$
    * The last line transform the standardized $z_x$ to the original scale $x$
    * Recall that data is standardized as follows
    $$
    z_i = \frac{x_i - \mu_X}{\sigma_X}
    $$
    * To bring the data to the original scale we simply solve for $x_i$ (undo the computation)
    $$
    x_i = \mu_X + z_i\sigma_X
    $$
    * In our example, $\mu_X = 0$ and $\sigma_X = \exp(v)$
    

In [ ]:
with pm.Model() as m13_8:
    v = pm.Normal('v',mu=0,sigma=3)
    z = pm.Normal('z',mu=0,sigma=1) # standardized `x`
    x = pm.Deterministic('x',z*pm.math.exp(v)) # non-standardized x: multiply by its standard deviation and add expected value (0 in this example)

In [ ]:
trace_13_8 = pm.sample(model=m13_8,return_inferencedata=True)

* Note the lack of divergences and good diagnosis below 😎

In [ ]:
pm.summary(trace_13_8)

In [ ]:
pm.plot_trace(trace_13_8);

In [ ]:
pm.plot_pair(trace_13_8,var_names=['x','v'],divergences=True);

* The plot compares the behaviour of a simulation in the centered (left) and non-centered (right) models
    * The lines show the high-density areas

<img src="figs/divergences-nc-c.png" width=600px>

### Non-centered chimpanzees

* We will try to fix the divergences in the chimpanzees models

<br>

* First, we try increasing the target accept of the HMC algorithm
    * This solution reduces the step size and prevents the simulation from falling to far from the actual log density function
    * It is also recommended in the PyMC divergences message
    * It is an easy solution (that rarely works effectively 😅, personal experience)

<br>

In [ ]:
trace_13_4_ta = pm.sample(model=m13_4,target_accept=.99)

* Note that the number of divergences has dropped considerably
    * Compare number of divergences with [lower target accept version of the model](#divergences-chimpanzees)
 

<div style="width:400px;  height: 50px; display:flex; align-items:center; justify-content:center; background:blue; text-align:center; box-sizing:border-box; margin:8px auto; color: white">
    <strong style="font-size: 16px">
        Why does increasing the target_accept parameter may help to reduce divergences?
    </strong>
</div>

In [ ]:
pm.summary(trace_13_4_ta)

* The non-centered version of the model is defined as follows

\begin{align*}
L_i &\sim \mathrm{Binomial}(1,p_i) \\
\mathrm{logit}(p_i) &= \underbrace{\bar{\alpha} + z_{\mathrm{ACTOR}[i]}\sigma_\alpha}_{\alpha_{\mathrm{ACTOR}[i]}} + 
\underbrace{x_{ \mathrm{{BLOCK}[i]} } \sigma_\gamma}_{\gamma_{\mathrm{BLOCK}[i]}} + 
\beta_{\mathrm{TREATMENT}[i]}\\
\beta_j &\sim \mathrm{Normal}(0,0.5) ~~ \text{ for }j \in \{1,\ldots,4\} \\
z_k &\sim \mathrm{Normal}(0,1) ~~~~~ \text{ for }k \in \{1,\ldots,7\} \\
x_l &\sim \mathrm{Normal}(0,1) ~~~ \,\,\, \text{ for }l \in \{1,\ldots,6\} \\
\bar{\alpha} &\sim \mathrm{Normal}(0,1.5) \\
\sigma_\alpha &\sim \mathrm{Exponential}(1) \\
\sigma_\gamma &\sim \mathrm{Exponential}(1) 
\end{align*}

In [ ]:
with pm.Model() as m13_4_nc:
    σ_α = pm.Exponential('σ_α',lam=1.0)
    σ_γ = pm.Exponential('σ_γ',lam=1.0)
    α_bar = pm.Normal('α_bar',mu=0,sigma=1.5)
    
    z_γ = pm.Normal('z_γ',mu=0,sigma=1,shape=Nblock)
    z_α = pm.Normal('z_α',mu=0,sigma=1,shape=Nactor)       
    
    β = pm.Normal('β',mu=0,sigma=0.5,shape=Ntreatments)
    
    #                      ____________α___________   _______γ________
    p = pm.math.invlogit( (α_bar + z_α[actor]*σ_α) + (z_γ[block]*σ_γ) + β[treatment])
    
    α = pm.Deterministic('α', α_bar + z_α*σ_α)
    γ = pm.Deterministic('γ', α_bar + z_γ*σ_γ)
    
    L = pm.Binomial('L',n=1,p=p,observed=d_ch.pulled_left)

In [ ]:
pm.model_to_graphviz(m13_4_nc)

In [ ]:
trace_13_4_nc = pm.sample(model=m13_4_nc)

* We compare the ESS values for each parameter between the two models

<br>

* We clearly observe that almost all parameters have higher ESS values in the non-centered model
    * Points are located above dashed line

In [ ]:
m13_4_summ = pm.summary(trace_13_4, kind="diagnostics", round_to=2)["ess_bulk"]
m13_4_summ.name = "m13_4"

m13_4nc_summ = pm.summary(trace_13_4_nc, var_names=["~z_γ", "~z_α"], kind="diagnostics", round_to=2)[
    "ess_bulk"
]
m13_4nc_summ.name = "m13_4nc"

ess_bulk = pd.concat([m13_4_summ, m13_4nc_summ], axis=1, sort=True)

In [ ]:
plt.plot(ess_bulk.m13_4.values, ess_bulk.m13_4nc.values, "o", alpha=0.5)

max_val = ess_bulk.m13_4nc.max() + 100
plt.plot(np.arange(max_val), np.arange(max_val), "k--", alpha=0.6)

plt.xlabel("n_eff (centered)")
plt.xlim(0, max_val)

plt.ylabel("n_eff (non-centered)")
plt.ylim(0, max_val);

## Posterior predictions in multilevel models

* Generating posterior prediction samples in multilevel follows the same principle as in single-level models
    * It is just a bit more tedious due to the number of parameters and structure of the model

<br>

    
* In multilevel models, we do not expect the posterior predictive samples to perfectly match the data
    * Remember that high-level parameters pull lower parameters closer to each other
    
<br>


* Below we discuss how to generate posterior predictive samples for the probability of success for different treatments

In [ ]:
# we use the centered model with `pm.Data` elements to help generating the posterior predictive samples
with pm.Model() as m13_4_data:
    σ_α = pm.Exponential('σ_α',lam=1.0)
    σ_γ = pm.Exponential('σ_γ',lam=1.0)
    α_bar = pm.Normal('α_bar',mu=0,sigma=1.5)
    
    γ = pm.Normal('γ',mu=0,sigma=σ_γ,shape=Nblock)
    α = pm.Normal('α',mu=α_bar,sigma=σ_α,shape=Nactor)
    β = pm.Normal('β',mu=0,sigma=0.5,shape=Ntreatments)
    
    # we only add this data blocks, to later be able to define their value in the posterior predictive check
    actor_ = pm.Data('actor',actor)
    block_ = pm.Data('block',block)
    treatment_ = pm.Data('treatment',treatment)
    
    p = pm.math.invlogit(α[actor_] + γ[block_] + β[treatment_])
    
    L = pm.Binomial('L',n=1,p=p,observed=d_ch.pulled_left)

In [ ]:
trace_13_4_data = pm.sample(model=m13_4_data)

### Posterior predictions for a specific chimpanzee


* We generate posterior predictive data for chimpanzee 2 

In [ ]:
# renaming for convenience when selecting samples
post = trace_13_4_data.posterior.rename_dims(
    {"α_dim_0": "actor", "γ_dim_0": "block_id", "β_dim_0": "treatment"}
)

* This function generates probability samples of pulling left of an actor and block

In [ ]:
def p_link(actor, block_id, post):
    post_filtered = post.sel(actor=actor,block_id=block_id)
    logodds = post_filtered["α"] + post_filtered["γ"] + post_filtered["β"]
    return logistic(logodds)

* We compute and plot the data for chimpanzee 2 (actor 1) in the first block (block 0)

In [ ]:
chimp = 2
block = 1
p_raw = p_link(actor=chimp-1, block_id=block-1, post=post)
p_raw.mean(dim=("chain", "draw")).values.round(2), pm.hdi(p_raw.values,hdi_prob=.89).round(2)

In [ ]:
# taken from pymc github
def chimp_pp_plot(hdi_data, mean_data, title):
    _, ax = plt.subplots(1, 1, figsize=(5, 5))
    pm.plot_hdi(range(4), hdi_data, hdi_prob=.89)
    ax.plot(mean_data)

    ax.set_ylim(0, 1.1)
    ax.set_xlabel("treatment")
    ax.set_ylabel("proportion pulled left")
    ax.set_xticks(range(4), ("R/N", "L/N", "R/P", "L/P"))
    plt.title(title);

In [ ]:
chimp_pp_plot(
    hdi_data=p_raw.values,
    mean_data=p_raw.mean(dim=("chain", "draw")).values,
    title=f"Posterior predictions for chimpanzee #{chimp}",
)

### Posterior predictions for the population of chimpanzees

* The same process can be repeated for the population parameters    
    * We simply need to sample for a different part of the model 

<br>

* We simply need to replace $\alpha_{\mathrm{ACTOR}[1]}$ by $\bar{\alpha}$
    * We also ignore the block parameter, as we extrapolate to new blocks
        * This also means that we assume that the effect of block is 0 (which was suggested by our previous analysis)

<br>
 
* These are posterior predictions for an average chimpanzee

In [ ]:
p_raw = logistic(post["α_bar"] + post["β"])

chimp_pp_plot(
    hdi_data=p_raw.data,
    mean_data=p_raw.mean(dim=("chain", "draw")).data,
    title="average actor",
)

### Variation among actors


* The results above were targeted to:
    * Chimpanzee 2, block 1
    * Average of the population

<br>


* This makes it hard to appreciate the variance in the complete population

<br>

* Below we add the $\sigma_\alpha$ parameter to account for sampled population variance

<br>

* This change shows results for chimpanzees not included int the data (as it is a population level parameter)

In [ ]:
α_sim = np.random.normal(loc=post["α_bar"],scale=post['σ_α'])
p_raw_asim = logistic(α_sim[:,:,None] + post["β"])

chimp_pp_plot(
    hdi_data=p_raw_asim.data,
    mean_data=p_raw_asim.mean(dim=("chain", "draw")).data,
    title="marginal of actor",
)

* Plotting each line separately we can better observe each the simulated chimpanzees

In [ ]:
p_raw_asim_st = p_raw_asim.stack(sample=("chain", "draw"))

_, ax = plt.subplots(1, 1, figsize=(5, 5))
ax.plot(np.tile(np.arange(4), (100, 1)).T, p_raw_asim_st[:, :100].data, "k", alpha=0.4)

ax.set_ylim(0, 1.1)
ax.set_xlabel("treatment")
ax.set_ylabel("proportion pulled left")
ax.set_xticks(range(4), ("R/N", "L/N", "R/P", "L/P"))
plt.title("simulated actors");